In [25]:
#0
from google.colab import drive
drive.mount('/content/drive')


In [26]:
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [27]:
DATA_PATH = "/content/drive/MyDrive/EEG_TSSL_DATA"

windows = np.load(os.path.join(DATA_PATH, "windows.npy"))
labels = np.load(os.path.join(DATA_PATH, "labels_windows.npy"))
trial_ids = np.load(os.path.join(DATA_PATH, "trial_ids.npy"))
bandpower_features = np.load(os.path.join(DATA_PATH, "bandpower_features.npy"))

print("Windows:", windows.shape)
print("Bandpower:", bandpower_features.shape)
print("Labels:", labels.shape)

Windows: (78080, 32, 256)
Bandpower: (78080, 128)
Labels: (78080,)


In [28]:
unique_trials = np.unique(trial_ids)
trial_labels = np.array([labels[trial_ids == t][0] for t in unique_trials])

train_trials, test_trials = train_test_split(
    unique_trials,
    test_size=0.3,
    stratify=trial_labels,
    random_state=42
)

train_mask = np.isin(trial_ids, train_trials)
test_mask  = np.isin(trial_ids, test_trials)

X_train = bandpower_features[train_mask]
y_train = labels[train_mask]

X_test = bandpower_features[test_mask]
y_test = labels[test_mask]

trial_test_ids = trial_ids[test_mask]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (54656, 128)
Test: (23424, 128)


In [29]:
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [30]:
batch_size = 64

train_loader = DataLoader(EEGDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(EEGDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

In [31]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

num_classes = len(np.unique(labels))
model = MLP(input_dim=bandpower_features.shape[1], num_classes=num_classes)

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [33]:
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

Epoch 1/15, Loss: 842.5457
Epoch 2/15, Loss: 820.6184
Epoch 3/15, Loss: 807.8166
Epoch 4/15, Loss: 798.5973
Epoch 5/15, Loss: 791.5313
Epoch 6/15, Loss: 785.9291
Epoch 7/15, Loss: 781.1140
Epoch 8/15, Loss: 776.0009
Epoch 9/15, Loss: 771.4534
Epoch 10/15, Loss: 766.5170
Epoch 11/15, Loss: 761.5516
Epoch 12/15, Loss: 759.2317
Epoch 13/15, Loss: 754.9133
Epoch 14/15, Loss: 750.6621
Epoch 15/15, Loss: 746.7440


In [34]:
model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        outputs = model(X)
        preds = outputs.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(y.numpy())

print("Window Accuracy:", accuracy_score(all_true, all_preds))

Window Accuracy: 0.5557547814207651


In [35]:
from collections import defaultdict

trial_preds = defaultdict(list)
trial_true = {}

for pred, true, tid in zip(all_preds, all_true, trial_test_ids):
    trial_preds[tid].append(pred)
    trial_true[tid] = true

final_preds = []
final_true = []

for tid in trial_preds:
    final_preds.append(np.bincount(trial_preds[tid]).argmax())
    final_true.append(trial_true[tid])

print("Trial Accuracy:", accuracy_score(final_true, final_preds))
print("Confusion Matrix:\n", confusion_matrix(final_true, final_preds))

Trial Accuracy: 0.5651041666666666
Confusion Matrix:
 [[196   7  11]
 [ 76   9   4]
 [ 66   3  12]]


In [36]:
# ===== Save encoder after Temporal + Frequency SSL (Notebook-2) =====
save_path = "/content/drive/MyDrive/EEG_TSSL_DATA/encoder_cnn_transformer_tfssl.pt"

model.eval()  # optional but clean
torch.save(model.state_dict(), save_path)

print(f"[Notebook-2] Encoder saved to: {save_path}")

[Notebook-2] Encoder saved to: /content/drive/MyDrive/EEG_TSSL_DATA/encoder_cnn_transformer_tfssl.pt
